# Winam Gulf Water Quality & Algal Bloom Monitoring (2018–2024)
### A Remote Sensing Framework using Sentinel-2, Landsat 8, and Google Earth Engine

**Author:** Gerald Omollo — GIS Analyst & Remote Sensing Specialist, Dedan Kimathi University of Technology (DeKUT)

**Objective:** Assess spatiotemporal eutrophication and algal bloom dynamics in Winam Gulf, Lake Victoria, by deriving
water quality proxies (Chlorophyll-a, TSS, CDOM) from satellite imagery, classifying bloom intensity, mapping
persistent hotspots, evaluating spatial correspondence with discharge points, and testing environmental drivers.

## Structure

**Part I — Core Pipeline**
1. Authentication & Environment Setup
2. Study Area — Data-Derived AOI (no manual bounding box)
3. Data Acquisition (Sentinel-2 MSI & Landsat 8 OLI)
4. Preprocessing — Cloud Masking, Water Masking, Sun-Glint Correction
5. Water Quality Proxy Derivation (Chl-a, TSS, CDOM)
6. Monthly Time-Series Compilation (2018–2024)
7. Bloom Intensity Classification (K-means, severity-ranked)
8. Hotspot Analysis (Bloom Frequency Raster)
9. Discharge Point Overlay (data-loading stub — needs your NEMA data)

**Part II — Enhanced Analysis**
- **Tier 1:** Reference/control site comparison · Rainfall-bloom correlation · Distance-decay regression
- **Tier 2:** Mann-Kendall trend test · Catchment land cover change · Literature-referenced risk classification
- **Tier 3:** Seasonal bloom forecast · Auto-generated one-page monitoring report

**Part III** — Interpretation guidance, exports, and an honest flags table.

> **Read this before running anything:** every cell that depends on data I can't independently verify (NEMA
> discharge points, literature threshold values, your GEE project ID) is explicitly marked as a stub or flagged
> with the specific number you need to supply. Nothing below is silently fabricated — where I don't have a real
> number, I've said so rather than inventing one that looks plausible.


# Part I — Core Pipeline

## 1. Authentication & Environment Setup

Requires a registered Google Cloud project with the Earth Engine API enabled
(https://code.earthengine.google.com/register). Replace the project ID below with your own.


In [1]:
# !pip install earthengine-api geemap geopandas rasterio scikit-learn scipy statsmodels -q

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from datetime import datetime

ee.Authenticate()
ee.Initialize(project='your-gee-project-id')  # <-- replace with your Cloud project ID
print("Earth Engine initialized successfully.")


EEException: Caller does not have required permission to use project your-gee-project-id. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=your-gee-project-id and then retry. Propagation of the new permission may take a few minutes.

## 2. Study Area — Data-Derived AOI

**What changed from the earlier version:** instead of a hand-drawn rectangular bounding box (which risked including
open-lake pixels that dilute the gulf-specific bloom signal), the AOI here is **derived from real water-occurrence
data** — the JRC Global Surface Water dataset — clipped at the Rusinga Channel, the actual named strait connecting
Winam Gulf to the main body of Lake Victoria.

**How it works:**
1. Define a generous *search region* (not the final AOI) around the known gulf location.
2. Use JRC Global Surface Water `occurrence` band to isolate pixels that are water >75% of the observed record
   (1984–2021) — i.e., permanent water, not seasonal wetland or misclassified land.
3. Clip the search region at the Rusinga Channel using a cutline near Mbita/Rusinga Island, where the gulf narrows
   to meet the open lake — this is the real geographic feature that separates the gulf from the main lake body.
4. Vectorize the resulting water mask and keep the largest connected polygon (i.e., the gulf itself, not scattered
   small water bodies).

**⚠️ Honest limitation:** the Rusinga Channel cutline coordinates below are placed from general geographic
knowledge of the Mbita/Rusinga Island area, not a surveyed boundary file — I could not locate a publicly downloadable,
Winam-Gulf-specific shapefile to embed directly (Lake Victoria-wide shapefiles exist, e.g. via ICPAC/RCMRD
geoportals, but require manual download and licensing acceptance through a browser, so I can't pull them
programmatically here). **Visually inspect the rendered polygon in the map output below before trusting it** —
if the cutline is misplaced, the derived polygon will either bleed into the open lake or cut off part of the gulf,
and you'll be able to see that immediately on the map. If you have access to an authoritative shapefile (ICPAC,
RCMRD, or LVBC), loading it directly (Option B, commented below) is strictly better than this derived approach.


In [ ]:
# --- Generous search region (intentionally loose — NOT the final AOI) ---
search_region = ee.Geometry.Rectangle([34.30, -0.55, 35.05, 0.15])

# --- Rusinga Channel cutline (approximate — validate visually, see markdown above) ---
# Cuts the search region at the Mbita/Rusinga Island strait, separating gulf water from open lake
cutline_lon = 34.22  # approximate longitude of the channel constriction near Mbita
gulf_side = ee.Geometry.Rectangle([cutline_lon, -0.55, 35.05, 0.15])

# --- JRC Global Surface Water: permanent water mask ---
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
permanent_water = gsw.select('occurrence').gt(75)  # >75% of observations = permanent water

water_in_search = permanent_water.clip(gulf_side).selfMask()

# --- Vectorize and keep the largest polygon (the gulf itself) ---
water_vectors = water_in_search.reduceToVectors(
    geometry=gulf_side,
    scale=100,           # coarse scale for vectorization speed — fine for AOI delineation, not analysis
    geometryType='polygon',
    eightConnected=True,
    maxPixels=1e10,
    bestEffort=True
)

# Select the polygon with the largest area (filters out small disconnected water bodies)
water_vectors = water_vectors.map(lambda f: f.set('area', f.geometry().area(maxError=100)))
aoi_feature = water_vectors.sort('area', False).first()
aoi = aoi_feature.geometry()

print("Derived AOI area (km²):", aoi.area(maxError=100).divide(1e6).getInfo())

# --- Option B: if you obtain an authoritative shapefile, use this instead of the derived AOI ---
# gulf_gdf = gpd.read_file('winam_gulf_boundary.shp')
# aoi = geemap.geopandas_to_ee(gulf_gdf).geometry()

# --- Visual validation — DO NOT skip this check ---
Map = geemap.Map(center=[-0.15, 34.65], zoom=9)
Map.addLayer(search_region, {'color': 'gray'}, 'Search region (loose)', False)
Map.addLayer(permanent_water.clip(search_region).selfMask(), {'palette': ['00BFFF']}, 'All permanent water in search region')
Map.addLayer(aoi, {'color': 'red'}, 'Derived Winam Gulf AOI — INSPECT THIS')
Map.addLayerControl()
Map


## 3. Data Acquisition — Sentinel-2 MSI & Landsat 8 OLI

Both collections used are already atmospherically corrected surface-reflectance products (Sen2Cor for S2,
Collection 2 Level-2 for L8) — no separate atmospheric correction step needed.


In [ ]:
START_DATE = '2018-01-01'
END_DATE   = '2024-12-31'
CLOUD_THRESHOLD = 40

s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(aoi)
                  .filterDate(START_DATE, END_DATE)
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD)))

l8_collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                  .filterBounds(aoi)
                  .filterDate(START_DATE, END_DATE)
                  .filter(ee.Filter.lt('CLOUD_COVER', CLOUD_THRESHOLD)))

print("Sentinel-2 scenes found:", s2_collection.size().getInfo())
print("Landsat 8 scenes found:", l8_collection.size().getInfo())


## 4. Preprocessing — Cloud Masking, Water Masking, Sun-Glint Correction

Cloud masking via SCL, water masking via NDWI, and sun-glint correction using the **regression-based Hedley et
al. (2005) method** — an upgrade from a fixed-slope approximation.

**How Hedley deglinting actually works, and why it's better:**
1. Sample a set of water pixels spanning a range of glint intensity within the same scene.
2. For each visible band, regress its reflectance against NIR reflectance across those samples: `R_i = a_i·NIR + b_i`.
   The slope `a_i` captures how strongly glint in that band tracks NIR brightness — this is scene-specific and
   band-specific, which the earlier fixed-slope subtraction couldn't represent.
3. Deglinted reflectance: `R_i' = R_i − a_i·(NIR − NIR_min)`, where `NIR_min` is the minimum NIR value in the
   sample (an ambient/zero-glint reference).

**⚠️ Honest limitation carried over from the earlier version:** this still assumes a locally uniform glint field
and a reasonably homogeneous water substrate within the sampled area — both real assumptions in a shallow,
sediment-influenced gulf. It is a substantially better approximation than the fixed-slope version, but is still
not equivalent to careful manual glint-area selection in ENVI. If a1 slope values regress to unrealistic values
(e.g., negative, or >>1), check the sample and fall back to the simplified method with that flagged in your
methods.


In [ ]:
def mask_s2_clouds(image):
    scl = image.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

def mask_water_ndwi(image, threshold=0.0):
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    return image.updateMask(ndwi.gt(threshold)).addBands(ndwi)

def deglint_hedley(image, region, sample_scale=20, n_samples=500, seed=42):
    """Regression-based sun-glint correction (Hedley et al., 2005)."""
    glint_bands = ['B2', 'B3', 'B4']
    nir = image.select('B8')

    sample = image.select(glint_bands + ['B8']).sample(
        region=region, scale=sample_scale, numPixels=n_samples, seed=seed, geometries=False
    )

    nir_min = ee.Number(sample.aggregate_min('B8'))
    corrected = image

    for b in glint_bands:
        fit = sample.reduceColumns(
            reducer=ee.Reducer.linearFit(), selectors=['B8', b]
        )
        slope = ee.Number(fit.get('scale'))
        # Guard against degenerate/negative slopes — glint correction should never invert the band
        slope = slope.max(0)
        corrected_band = image.select(b).subtract(slope.multiply(nir.subtract(nir_min))).max(0).rename(b)
        corrected = corrected.addBands(corrected_band, overwrite=True)

    return corrected

def deglint_simple_fallback(image):
    """Fixed-slope fallback — only use if deglint_hedley's regression looks degenerate (see markdown above)."""
    nir = image.select('B8')
    corrected = image
    for b in ['B2', 'B3', 'B4']:
        corrected_band = image.select(b).subtract(nir).max(0).rename(b)
        corrected = corrected.addBands(corrected_band, overwrite=True)
    return corrected

def preprocess_s2(image):
    img = mask_water_ndwi(mask_s2_clouds(image))
    img = deglint_hedley(img, aoi)
    return img

s2_processed = s2_collection.map(preprocess_s2)
print("Preprocessing pipeline applied (Hedley regression-based deglinting).")


## 5. Water Quality Proxy Derivation

| Proxy | Index | Formula | Interpretation |
|---|---|---|---|
| Chlorophyll-a | NDCI | (B5 − B4) / (B5 + B4) | Higher = more algal biomass |
| TSS | Red-NIR ratio | (B4 − B3) / (B4 + B3) | Higher = more suspended sediment |
| CDOM | Blue/Green ratio | B2 / B3 | Higher ratio ≈ lower CDOM |

**⚠️ These are relative indices, not calibrated concentrations** — don't report as absolute mg/m³ without an
in-situ regression fit.


In [ ]:
def add_water_quality_indices(image):
    ndci = image.normalizedDifference(['B5', 'B4']).rename('NDCI_chla')
    tss = image.normalizedDifference(['B4', 'B3']).rename('TSS_proxy')
    cdom = image.select('B2').divide(image.select('B3')).rename('CDOM_proxy')
    return image.addBands([ndci, tss, cdom])

s2_with_indices = s2_processed.map(add_water_quality_indices)
print("Water quality indices added.")


### 5b. In-Situ Calibration — NDCI to Real Chl-a Concentration (mg/m³)

**Why this matters:** everything up to this point produces a *relative* index — you can say "this month is higher
than that month" but not "this is 40 mg/m³." That single limitation gets flagged repeatedly through this notebook.
This section fixes it, partially, using a real, publicly available dataset.

**Dataset:** Deirmendjian et al. (2020), *"Dataset for 'Limnological changes in Lake Victoria since the
mid-20th century'"*, Zenodo, `https://doi.org/10.5281/zenodo.4247352` — paired with Deirmendjian et al. (2021),
*Freshwater Biology* 66(8), 1630–1647. Contains 219 station-visits across nearshore (8–31 m depth) and offshore
(29–73 m depth) sites, including measured Chlorophyll-a, collected March/June/October 2018–2019.

**⚠️ Read this before treating the resulting regression as gulf-specific:** these stations are in **Ugandan
waters** — nearshore bays and offshore open-lake sites — not Winam Gulf itself. This is a **regional Lake
Victoria calibration, not a local one.** Using it assumes the NDCI–Chl-a relationship is reasonably stable across
Lake Victoria's optically similar nearshore zones, which is a real assumption, not a guarantee — Winam Gulf's
higher turbidity/CDOM regime (per Ogembo & Mohamed's 33–50 mg/m³ reading) may shift the true local relationship.
Report the calibration's R²/RMSE honestly, and state explicitly in your methods that it is a **regional proxy
calibration**, not a Winam-Gulf in-situ fit — that distinction matters if a reviewer checks your source.

**What this cell does:** downloads the public dataset, matches station coordinates/dates to the nearest available
Sentinel-2 NDCI value (±7 day window), fits a linear regression, and reports fit statistics you should include in
your methods section (or discard the approach, honestly, if the fit is poor — see the fit-quality check below).


In [ ]:
import requests
from io import BytesIO

# Zenodo doesn't publish a single flat CSV at a fixed path across versions — fetch the record's file
# listing first, then download the concentration/chlorophyll file. Adjust file selection if the record's
# file names change in a future dataset version.
ZENODO_RECORD_ID = '4247352'
zenodo_api_url = f'https://zenodo.org/api/records/{ZENODO_RECORD_ID}'

try:
    record = requests.get(zenodo_api_url, timeout=30).json()
    files = record.get('files', [])
    print("Files in Zenodo record:")
    for f in files:
        print(" -", f.get('key'))
except Exception as e:
    print(f"⚠️ Could not reach Zenodo from this environment: {e}")
    print("If this fails in Colab too (rather than just a sandboxed environment), download manually from")
    print("https://doi.org/10.5281/zenodo.4247352 and load the CSV with pd.read_csv() instead.")

# ⚠️ MANUAL STEP REQUIRED: inspect the printed file list above, identify the file containing
# station lat/lon, date, and Chl-a measurements, and load it explicitly — column names and file
# structure in the actual archive were not verified from within this notebook-building process.
# Example (adjust filename/columns to match what's actually in the archive):
#
# insitu_df = pd.read_csv(files[i]['links']['self'])
# insitu_df = insitu_df.rename(columns={'actual_lat_col': 'lat', 'actual_lon_col': 'lon',
#                                         'actual_date_col': 'date', 'actual_chla_col': 'chla_mgm3'})

print("\n⚠️ Do not proceed to the regression cell below until insitu_df is loaded with verified column names —")
print("guessing column names against unseen data risks silently misaligning coordinates or units.")


In [ ]:
def match_insitu_to_satellite(insitu_df, s2_collection_with_ndci, date_window_days=7):
    """Matches each in-situ station to the nearest-date Sentinel-2 NDCI value at that point."""
    matched_records = []
    for _, row in insitu_df.iterrows():
        point = ee.Geometry.Point([row['lon'], row['lat']])
        target_date = pd.to_datetime(row['date'])
        window_start = (target_date - pd.Timedelta(days=date_window_days)).strftime('%Y-%m-%d')
        window_end = (target_date + pd.Timedelta(days=date_window_days)).strftime('%Y-%m-%d')

        nearby_imgs = s2_collection_with_ndci.filterBounds(point).filterDate(window_start, window_end)
        if nearby_imgs.size().getInfo() == 0:
            continue

        ndci_val = nearby_imgs.select('NDCI_chla').median().reduceRegion(
            reducer=ee.Reducer.mean(), geometry=point.buffer(30), scale=20, bestEffort=True
        ).get('NDCI_chla')
        ndci_val = ndci_val.getInfo() if ndci_val is not None else None

        if ndci_val is not None:
            matched_records.append({'date': target_date, 'lat': row['lat'], 'lon': row['lon'],
                                     'chla_mgm3': row['chla_mgm3'], 'NDCI_matched': ndci_val})

    return pd.DataFrame(matched_records)

# matched_df = match_insitu_to_satellite(insitu_df, s2_with_indices)
#
# from scipy import stats
# slope, intercept, r_value, p_value, std_err = stats.linregress(matched_df['NDCI_matched'], matched_df['chla_mgm3'])
# print(f"Chl-a (mg/m3) = {slope:.2f} * NDCI + {intercept:.2f}")
# print(f"R2 = {r_value**2:.3f}, n = {len(matched_df)}")
#
# ⚠️ FIT-QUALITY GATE — do not use this regression in your results unless BOTH hold:
# 1. R2 > 0.5 (below this, Kapalanga et al. 2020's own acceptability threshold, cited in the Gimode paper,
#    is not met — the fit isn't explaining enough variance to trust)
# 2. n >= 10 matched points (fewer than this and the fit is not meaningfully more reliable than
#    the uncalibrated NDCI proxy you already have)
#
# If either check fails: report that calibration was attempted and did not meet a defensible threshold,
# and continue reporting relative NDCI values with the existing 'not a calibrated concentration' caveat
# rather than presenting a weak regression as if it were reliable.

print("match_insitu_to_satellite() defined. Requires insitu_df from the previous cell before running.")


### 5c. Landsat 8 Indices & Sensor Cross-Calibration

**Why this section exists:** Landsat 8 was pulled in Section 3 but, until now, never actually used — flagged
earlier as a gap. Bringing it in for temporal gap-filling without checking sensor agreement first risks reading
a sensor-transition artifact as a real water-quality change.

**⚠️ A real constraint, not a workaround:** Landsat 8 OLI has no red-edge band (~705 nm), which NDCI requires
(it needs Sentinel-2's B5). **NDCI/Chl-a cannot be computed from Landsat 8 at all** — this isn't a calibration
problem, it's a missing-band problem, and no regression fixes it. TSS and CDOM proxies use only blue/green/red/NIR,
which both sensors have, so cross-calibration below applies **only to TSS_proxy and CDOM_proxy**, not to the
Chl-a/NDCI series. If you gap-fill months using L8, you'll be filling TSS/CDOM gaps only — the Chl-a time series
stays Sentinel-2-only by necessity.


In [ ]:
def mask_l8_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_bit, shadow_bit, cirrus_bit = 1 << 3, 1 << 4, 1 << 2
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(shadow_bit).eq(0)).And(qa.bitwiseAnd(cirrus_bit).eq(0))
    # Collection 2 L2 scale factors (per USGS documentation) to convert DN to reflectance
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return optical.updateMask(mask).copyProperties(image, ['system:time_start'])

def add_l8_water_quality_indices(image):
    # L8 SR band mapping: SR_B2=blue, SR_B3=green, SR_B4=red, SR_B5=NIR (NO red-edge band — no NDCI possible)
    tss = image.normalizedDifference(['SR_B4', 'SR_B3']).rename('TSS_proxy')
    cdom = image.select('SR_B2').divide(image.select('SR_B3')).rename('CDOM_proxy')
    ndwi = image.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
    return image.addBands([tss, cdom, ndwi])

def preprocess_l8(image):
    img = mask_l8_clouds(image)
    img = img.updateMask(img.normalizedDifference(['SR_B3', 'SR_B5']).gt(0))  # water mask
    return add_l8_water_quality_indices(img)

l8_processed = l8_collection.map(preprocess_l8)
print("Landsat 8 preprocessing applied — TSS/CDOM proxies only (no NDCI, see markdown above).")


In [ ]:
def cross_calibrate_sensors(s2_imgs, l8_imgs, region, band, max_date_diff_days=3, sample_scale=30, n_points=300):
    """Regresses L8 vs S2 values for the same index at near-simultaneous dates/locations, to check whether
    they can be safely combined into one time series without a sensor-transition artifact."""
    s2_list = s2_imgs.toList(s2_imgs.size())
    matched_pairs = []

    n_s2 = s2_list.size().getInfo()
    for i in range(n_s2):
        s2_img = ee.Image(s2_list.get(i))
        s2_date = ee.Date(s2_img.get('system:time_start'))

        close_l8 = l8_imgs.filterDate(
            s2_date.advance(-max_date_diff_days, 'day'), s2_date.advance(max_date_diff_days, 'day')
        )
        if close_l8.size().getInfo() == 0:
            continue

        l8_img = close_l8.first()
        sample_pts = ee.FeatureCollection.randomPoints(region, n_points, seed=42)

        s2_vals = s2_img.select(band).sampleRegions(collection=sample_pts, scale=sample_scale)
        l8_vals = l8_img.select(band).sampleRegions(collection=sample_pts, scale=sample_scale)

        s2_df = geemap.ee_to_df(s2_vals)
        l8_df = geemap.ee_to_df(l8_vals)
        if len(s2_df) > 0 and len(l8_df) > 0 and len(s2_df) == len(l8_df):
            matched_pairs.append(pd.DataFrame({'S2': s2_df[band], 'L8': l8_df[band]}))

    if not matched_pairs:
        print(f"No overlapping S2/L8 acquisitions found within {max_date_diff_days} days for '{band}'.")
        return None

    paired_df = pd.concat(matched_pairs, ignore_index=True).dropna()

    from scipy import stats
    slope, intercept, r_value, p_value, std_err = stats.linregress(paired_df['L8'], paired_df['S2'])
    print(f"[{band}] S2 = {slope:.3f} * L8 + {intercept:.3f}  (R2={r_value**2:.3f}, n={len(paired_df)})")

    if r_value**2 < 0.5 or len(paired_df) < 10:
        print(f"⚠️ Fit quality below threshold (R2>0.5, n>=10 required) — do NOT combine S2 and L8 for '{band}'")
        print("into one time series. Keep them as separate series, or report L8 gap-fill months as a distinct")
        print("data source in your figures rather than blending them silently.")
        return None

    return {'slope': slope, 'intercept': intercept, 'r2': r_value**2, 'n': len(paired_df)}

# Example (computationally heavy — runs a client round-trip per matched scene pair):
# tss_calibration = cross_calibrate_sensors(s2_with_indices, l8_processed, aoi, band='TSS_proxy')
# cdom_calibration = cross_calibrate_sensors(s2_with_indices, l8_processed, aoi, band='CDOM_proxy')
print("cross_calibrate_sensors() defined. Run before blending any L8 month into the TSS/CDOM time series —")
print("never combine sensors on the assumption they agree without checking first.")


### 5b. In-Situ Calibration Check (Optional but Strongly Recommended)

Every index used so far (NDCI, TSS proxy, CDOM proxy) is a **relative** proxy — none of it has been checked
against a single real water sample. This section fits a local regression between your satellite NDCI and
actual measured Chl-a concentrations, so you can report calibrated mg/m³ values with a stated accuracy
(R², RMSE) instead of relying on relative-index caveats throughout your results section.

**Data source:** Deirmendjian et al. (2020), *Dataset for "Limnological changes in Lake Victoria since the
mid-20th century."* Zenodo. `https://doi.org/10.5281/zenodo.4247352` — 219 in-situ observations (light,
physico-chemical, and Chl-a) from Lake Victoria, collected 2018–2019 across three seasons.

**⚠️ Read this before treating the result as gulf-specific:** these stations are in **Ugandan nearshore and
offshore waters**, not Winam Gulf (Kenya). Lake Victoria's bays vary in optical properties (turbidity, CDOM
load, depth), so a regression fit on Ugandan stations is a **lake-wide approximation, not a Winam-Gulf-specific
calibration**. Use it to get a defensible order-of-magnitude conversion and a stated uncertainty — not to claim
gulf-specific calibration accuracy. If you can pair this with even a handful of real Winam Gulf field samples
(e.g., from KMFRI or a supervised field trip), a local correction term becomes possible; without that, report
this regression's R²/RMSE honestly as a lake-wide, not site-specific, calibration.

**File format note:** the dataset is a single 152 MB Excel file. Sheet names and exact column headers aren't
documented on the Zenodo landing page, so the cell below downloads it and prints the sheet/column structure —
**inspect that output and set the column names in the config block before running the regression.** I have not
guessed at column names here; guessing wrong would silently produce a meaningless fit.


In [ ]:
import requests
import os

ZENODO_URL = "https://zenodo.org/records/4247352/files/Deirmendjian_et_al_2020_Ecological_Changes_Lake_Victoria.xlsx?download=1"
LOCAL_PATH = "deirmendjian_lake_victoria_2020.xlsx"

# --- Step 1: Download (large file — this may take a few minutes) ---
if not os.path.exists(LOCAL_PATH):
    print("Downloading ~152 MB dataset — this may take a few minutes...")
    r = requests.get(ZENODO_URL, stream=True)
    r.raise_for_status()
    with open(LOCAL_PATH, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download complete.")
else:
    print("Already downloaded.")

# --- Step 2: Inspect structure — DO NOT skip this, column names are not guessed ---
xls = pd.ExcelFile(LOCAL_PATH)
print("Sheet names:", xls.sheet_names)
for sheet in xls.sheet_names:
    preview = pd.read_excel(LOCAL_PATH, sheet_name=sheet, nrows=5)
    print(f"\n--- Sheet: {sheet} ---")
    print("Columns:", list(preview.columns))
    print(preview.head(2))


In [ ]:
# --- Step 3: SET THESE after inspecting the printed output above — do not run with placeholder guesses ---
CHOSEN_SHEET = None   # e.g. 'Data' — set from the sheet list printed above
LAT_COL = None         # e.g. 'Latitude'
LON_COL = None          # e.g. 'Longitude'
CHLA_COL = None          # e.g. 'Chl-a (ug/L)'
DATE_COL = None           # e.g. 'Date'

if CHOSEN_SHEET is None:
    print("⚠️ Set CHOSEN_SHEET, LAT_COL, LON_COL, CHLA_COL, DATE_COL above based on the printed structure,")
    print("then re-run this cell. Not doing so is safer than guessing column names on a 152 MB file.")
else:
    insitu_df = pd.read_excel(LOCAL_PATH, sheet_name=CHOSEN_SHEET)
    insitu_df = insitu_df[[LAT_COL, LON_COL, CHLA_COL, DATE_COL]].dropna()
    insitu_df.columns = ['lat', 'lon', 'chla_ugL', 'date']
    insitu_df['date'] = pd.to_datetime(insitu_df['date'])

    print(f"Loaded {len(insitu_df)} in-situ Chl-a observations with valid coordinates and dates.")

    winam_center_lon, winam_center_lat = 34.65, -0.10
    insitu_df['dist_deg'] = ((insitu_df['lon'] - winam_center_lon)**2 + (insitu_df['lat'] - winam_center_lat)**2)**0.5
    nearby = insitu_df.sort_values('dist_deg').head(20)
    print("\nClosest 20 stations to Winam Gulf (still likely far — dataset is Ugandan waters):")
    print(nearby[['lat', 'lon', 'chla_ugL', 'date', 'dist_deg']])


In [ ]:
# --- Step 4: Extract matching satellite NDCI for each in-situ station/date, then fit the regression ---
def get_ndci_at_point(lat, lon, date, window_days=5):
    point = ee.Geometry.Point([lon, lat])
    date_str = pd.Timestamp(date)
    start = (date_str - pd.Timedelta(days=window_days)).strftime('%Y-%m-%d')
    end = (date_str + pd.Timedelta(days=window_days)).strftime('%Y-%m-%d')

    imgs = s2_with_indices.filterBounds(point).filterDate(start, end)
    if imgs.size().getInfo() == 0:
        return None

    composite = imgs.median()
    val = composite.select('NDCI_chla').reduceRegion(
        ee.Reducer.first(), point.buffer(50), scale=20, bestEffort=True
    ).get('NDCI_chla')
    return val.getInfo() if val is not None else None

if 'insitu_df' in dir() and CHOSEN_SHEET is not None:
    calib_records = []
    for _, row in insitu_df.iterrows():
        ndci_val = get_ndci_at_point(row['lat'], row['lon'], row['date'])
        if ndci_val is not None:
            calib_records.append({'ndci': ndci_val, 'chla_ugL': row['chla_ugL']})

    calib_df = pd.DataFrame(calib_records).dropna()
    print(f"Matched {len(calib_df)} in-situ samples to a cloud-free NDCI value within the search window.")

    if len(calib_df) >= 5:
        from scipy.stats import linregress
        reg = linregress(calib_df['ndci'], calib_df['chla_ugL'])
        print(f"\nFitted regression: Chl-a (ug/L) = {reg.slope:.2f} * NDCI + {reg.intercept:.2f}")
        print(f"R2 = {reg.rvalue**2:.3f}, p-value = {reg.pvalue:.4f}")

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(calib_df['ndci'], calib_df['chla_ugL'], alpha=0.6)
        x_line = np.linspace(calib_df['ndci'].min(), calib_df['ndci'].max(), 50)
        ax.plot(x_line, reg.slope * x_line + reg.intercept, color='red', label=f"R2={reg.rvalue**2:.2f}")
        ax.set_xlabel('Satellite NDCI'); ax.set_ylabel('In-situ Chl-a (ug/L)')
        ax.set_title('NDCI vs. In-Situ Chl-a — Lake Victoria (Ugandan stations)')
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig('ndci_calibration.png', dpi=200)
        plt.show()

        print("\n⚠️ R2 above reflects a LAKE-WIDE fit on Ugandan-water stations, not a Winam-Gulf-specific")
        print("calibration. Report it as such. If R2 is weak (<0.5), do not apply this regression to convert")
        print("Winam Gulf NDCI values to concentrations — report NDCI as a relative index instead, and note")
        print("that no adequate local calibration was available as an honest, citable limitation.")
    else:
        print("⚠️ Fewer than 5 matched samples — not enough for a meaningful regression. Report NDCI as a")
        print("relative index only; do not force a calibration fit on too few points.")
else:
    print("Run Steps 2-3 first and set the column names before running this cell.")


## 6. Monthly Time-Series Compilation (2018–2024)

Reduces each monthly image collection to AOI-wide statistics per index, producing a tabular time series. Gap
months (no cloud-free water pixels) are reported as missing, not silently interpolated.

**⚠️ Accuracy fix — median alone hides your blooms.** Algal blooms are episodic (days, not weeks). A monthly
*median* composite is good for a stable baseline but will systematically smooth a short bloom spike toward the
month's typical value — meaning your peak severity estimates could be understated. This version computes **both**:
- **Median** composite/statistic — stable baseline, less noise-sensitive, good for trend analysis (Section 13).
- **Max** NDCI within the month — peak-detection signal, better for bloom/hotspot identification (Sections 7, 8).

Report which one you're using for which claim in your methods — conflating them (e.g. citing the max value as if
it were a typical month) would overstate how bad conditions usually are.


In [ ]:
def compile_monthly_series(region, label):
    months = pd.date_range(START_DATE, END_DATE, freq='MS')
    median_composites, records = [], []

    for m_start in months:
        m_end = m_start + pd.offsets.MonthEnd(1)
        monthly_imgs = s2_with_indices.filterDate(m_start.strftime('%Y-%m-%d'), m_end.strftime('%Y-%m-%d'))
        count = monthly_imgs.size().getInfo()
        if count == 0:
            continue

        median_composite = monthly_imgs.median().set('month', m_start.strftime('%Y-%m-%d'))
        median_composites.append(median_composite)

        # Median statistics (stable baseline)
        median_stats = median_composite.select(['NDCI_chla', 'TSS_proxy', 'CDOM_proxy']).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=region, scale=20, maxPixels=1e9, bestEffort=True
        ).getInfo()

        # Max NDCI within the month (peak/bloom detection — see markdown above)
        max_composite = monthly_imgs.select('NDCI_chla').max()
        max_ndci = max_composite.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=region, scale=20, maxPixels=1e9, bestEffort=True
        ).get('NDCI_chla')

        # Spatial std dev — a crude per-month uncertainty/heterogeneity signal, not a substitute for
        # proper validation, but better than reporting a bare mean with no spread information
        ndci_std = median_composite.select('NDCI_chla').reduceRegion(
            reducer=ee.Reducer.stdDev(), geometry=region, scale=20, maxPixels=1e9, bestEffort=True
        ).get('NDCI_chla')

        records.append({
            'month': m_start,
            'n_scenes': count,
            'NDCI_chla': median_stats.get('NDCI_chla'),
            'NDCI_chla_max': max_ndci.getInfo() if max_ndci is not None else None,
            'NDCI_chla_std': ndci_std.getInfo() if ndci_std is not None else None,
            'TSS_proxy': median_stats.get('TSS_proxy'),
            'CDOM_proxy': median_stats.get('CDOM_proxy'),
        })

    df = pd.DataFrame(records).set_index('month')
    print(f"[{label}] Compiled {len(df)} monthly composites out of {len(months)} possible months.")
    return median_composites, df

monthly_composites, ts_df = compile_monthly_series(aoi, 'Winam Gulf')

# ⚠️ Low-confidence flag: months built from very few cloud-free scenes are less reliable — don't treat
# a 1-scene month with the same confidence as an 8-scene month when interpreting results.
LOW_CONFIDENCE_SCENE_THRESHOLD = 2
ts_df['low_confidence'] = ts_df['n_scenes'] <= LOW_CONFIDENCE_SCENE_THRESHOLD
n_low_conf = ts_df['low_confidence'].sum()
if n_low_conf > 0:
    print(f"⚠️ {n_low_conf} of {len(ts_df)} months are built from ≤{LOW_CONFIDENCE_SCENE_THRESHOLD} scenes — "
          f"flag these as low-confidence in any figure or table, don't present them with equal weight.")

ts_df.head()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

ts_df['NDCI_chla'].plot(ax=axes[0], color='green', marker='o', markersize=3, label='Median (baseline)')
ts_df['NDCI_chla_max'].plot(ax=axes[0], color='darkred', marker='^', markersize=3,
                              linestyle='--', alpha=0.7, label='Max (peak/bloom signal)')
# Mark low-confidence months so a sparse-data point isn't read with the same weight as a well-sampled one
low_conf_months = ts_df[ts_df['low_confidence']].index
for m in low_conf_months:
    axes[0].axvspan(m, m + pd.offsets.MonthEnd(1), color='gray', alpha=0.15)
axes[0].set_ylabel('NDCI (Chl-a proxy)')
axes[0].set_title('Winam Gulf Water Quality Proxies — Monthly Time Series (2018–2024)\n(shaded = low-confidence months, ≤2 scenes)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

ts_df['TSS_proxy'].plot(ax=axes[1], color='saddlebrown', marker='o', markersize=3)
axes[1].set_ylabel('TSS proxy'); axes[1].grid(alpha=0.3)

ts_df['CDOM_proxy'].plot(ax=axes[2], color='steelblue', marker='o', markersize=3)
axes[2].set_ylabel('CDOM proxy (B2/B3)'); axes[2].set_xlabel('Date'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('winam_gulf_timeseries.png', dpi=200)
plt.show()

months_total = len(pd.date_range(START_DATE, END_DATE, freq='MS'))
missing = months_total - len(ts_df)
if missing > 0:
    print(f"⚠️ {missing} months had no usable cloud-free water pixels — gaps, not zeros.")


In [ ]:
# Mean vs. Max NDCI — shows what the median/mean composite alone would have hidden
fig, ax = plt.subplots(figsize=(12, 5))
ts_df['NDCI_chla_mean'].plot(ax=ax, color='green', label='Monthly mean NDCI', linewidth=1.5)
ts_df['NDCI_chla_max'].plot(ax=ax, color='darkred', label='Monthly max NDCI (peak bloom pixels)',
                              linewidth=1, linestyle='--')
ax.fill_between(ts_df.index, ts_df['NDCI_chla_mean'], ts_df['NDCI_chla_max'],
                 color='red', alpha=0.1, label='Gap hidden by using mean/median alone')
ax.set_ylabel('NDCI (Chl-a proxy)')
ax.set_title('Winam Gulf — Mean vs. Peak (Max) Bloom Signal by Month')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('winam_gulf_mean_vs_max.png', dpi=200)
plt.show()

gap = (ts_df['NDCI_chla_max'] - ts_df['NDCI_chla_mean']).mean()
print(f"Average gap between max and mean NDCI across all months: {gap:.3f}")
print("A large, consistent gap means a monthly median/mean composite would have systematically understated")
print("peak bloom severity — report max alongside mean in your results, not mean alone.")


## 7. Bloom Intensity Classification (K-means, severity-ranked)

K-means with k=4 (non-bloom / low / moderate / severe) — **a judgment call**, validated in Tier 2 below against
literature thresholds. Clusters are re-ranked by mean NDCI so labels are comparable across months (raw K-means
cluster IDs are arbitrary per run and NOT comparable without this step).


In [ ]:
def classify_bloom_intensity(image, n_clusters=4, sample_scale=30, seed=42):
    training = image.select('NDCI_chla').sample(region=aoi, scale=sample_scale, numPixels=2000, seed=seed)
    clusterer = ee.Clusterer.wekaKMeans(n_clusters).train(training)
    return image.addBands(image.select('NDCI_chla').cluster(clusterer).rename('bloom_class'))

def relabel_clusters_by_severity(classified_image, ndci_band='NDCI_chla', class_band='bloom_class', n_clusters=4):
    means = []
    for c in range(n_clusters):
        mean_val = classified_image.updateMask(classified_image.select(class_band).eq(c)) \
            .select(ndci_band).reduceRegion(ee.Reducer.mean(), aoi, 30, maxPixels=1e9, bestEffort=True).get(ndci_band)
        means.append((c, mean_val.getInfo() if mean_val is not None else -999))
    ranked = sorted(means, key=lambda x: x[1])
    remap_from = [c for c, _ in ranked]
    relabeled = classified_image.select(class_band).remap(remap_from, list(range(n_clusters))).rename('bloom_severity')
    return classified_image.addBands(relabeled)

if monthly_composites:
    example_classified = relabel_clusters_by_severity(classify_bloom_intensity(monthly_composites[-1]))
    Map2 = geemap.Map(center=[-0.15, 34.65], zoom=9)
    Map2.addLayer(example_classified.select('bloom_severity'),
                   {'min': 0, 'max': 3, 'palette': ['blue', 'green', 'yellow', 'red']}, 'Bloom Severity (example)')
    Map2.addLayerControl()
    Map2


## 8. Hotspot Analysis — Bloom Frequency Raster

Proportion of valid months each pixel was classified as the most severe bloom class — a continuous density
surface, functionally equivalent to (and more information-preserving than) point-based KDE on discrete bloom
occurrences.


In [ ]:
def compute_bloom_frequency(classified_images_list, severity_band='bloom_severity', severe_class=3):
    severe_masks = [img.select(severity_band).eq(severe_class).unmask(0) for img in classified_images_list]
    valid_masks = [img.select(severity_band).mask() for img in classified_images_list]
    severe_sum = ee.ImageCollection(severe_masks).sum()
    valid_sum = ee.ImageCollection(valid_masks).sum()
    return severe_sum.divide(valid_sum.max(1)).rename('bloom_frequency')

# Full run over all months — computationally heavy (one .getInfo() per month per cluster in relabeling);
# consider running as a batch/background task rather than interactively.
# relabeled_composites = [relabel_clusters_by_severity(classify_bloom_intensity(c)) for c in monthly_composites]
# bloom_freq_raster = compute_bloom_frequency(relabeled_composites)
print("compute_bloom_frequency() defined.")


## 9. Discharge Point Overlay (data-loading stub)

**Still requires your NEMA discharge point data** — not fabricated here. Load as CSV (`name, lat, lon`) or
GeoJSON/shapefile once you have it.


In [ ]:
# discharge_df = pd.read_csv('nema_discharge_points.csv')
# discharge_points = ee.FeatureCollection([
#     ee.Feature(ee.Geometry.Point([row.lon, row.lat]), {'name': row.name})
#     for row in discharge_df.itertuples()
# ])
print("Load discharge point data above before running Tier 1's distance-decay analysis (Section 12).")


# Part II — Enhanced Analysis

## Tier 1 — Analytical Gaps That Change Your Conclusions

These three additions turn descriptive mapping into testable claims: a baseline to compare against, a mechanism
test for *why* blooms occur when they do, and a quantitative (not just visual) test of the discharge-point
hypothesis.


### 10. Reference / Control Site Comparison

Without a baseline, "Winam Gulf has algal blooms" is a description, not a finding. This pulls the same NDCI/TSS/CDOM
series for an **open-water Lake Victoria reference site** — chosen off Mfangano Island, away from any gulf or
shoreline influence — so you can report bloom frequency and intensity *relative to* the open lake, not in isolation.


In [ ]:
# Reference site: open water near Mfangano Island, clearly outside the gulf/Rusinga Channel influence
reference_point = ee.Geometry.Point([33.75, -0.45])
reference_aoi = reference_point.buffer(3000)  # 3 km radius — open water, away from shoreline effects

ref_composites, ref_ts_df = compile_monthly_series(reference_aoi, 'Open-lake reference site')

# Side-by-side comparison
comparison_df = pd.DataFrame({
    'Winam_Gulf_NDCI': ts_df['NDCI_chla'],
    'Reference_NDCI': ref_ts_df['NDCI_chla']
}).dropna()

fig, ax = plt.subplots(figsize=(12, 5))
comparison_df['Winam_Gulf_NDCI'].plot(ax=ax, color='darkred', label='Winam Gulf')
comparison_df['Reference_NDCI'].plot(ax=ax, color='steelblue', label='Open-lake reference')
ax.set_ylabel('NDCI (Chl-a proxy)')
ax.set_title('Winam Gulf vs. Open-Lake Reference — Chlorophyll-a Proxy')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('gulf_vs_reference.png', dpi=200)
plt.show()

if len(comparison_df) > 0:
    ratio = comparison_df['Winam_Gulf_NDCI'].mean() / comparison_df['Reference_NDCI'].mean()
    print(f"Winam Gulf mean NDCI is {ratio:.2f}x the open-lake reference over the overlapping record.")
    print("⚠️ Report this ratio with the overlap period explicitly stated — don't extrapolate beyond it.")


### 11. Rainfall–Bloom Correlation (CHIRPS)

Tests whether Chl-a peaks align with rainy-season runoff (supporting a runoff-driven mechanism) or with dry-season
low flushing (supporting a retention-driven mechanism) — replacing a visual guess with an actual lagged
cross-correlation.


**⚠️ Update after literature check:** Ogembo & Mohamed (2023) — the same water body, same NDCI method —
report Winam Gulf Chl-a peaking in **January–February**, attributed to dry-season heat rather than rains-driven
runoff. That's the opposite of the "runoff pulse" mechanism this section was originally testing. Treat this as
a competing hypothesis, not a settled answer: run the lagged correlation below and compare your actual result
against both a runoff-driven pattern (positive correlation with rain, 0–2 month lag) and a
temperature/dry-season pattern (peak Chl-a coinciding with low rainfall months) before concluding either way.


In [ ]:
# Catchment region for rainfall — Nyando/Kano Plains draining into the gulf (approximate; refine with a
# real watershed delineation, e.g. HydroSHEDS basin polygons, if precision matters for your final analysis)
catchment_region = ee.Geometry.Rectangle([34.70, -0.30, 35.30, 0.10])

chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterBounds(catchment_region).filterDate(START_DATE, END_DATE)

rainfall_records = []
for m_start in pd.date_range(START_DATE, END_DATE, freq='MS'):
    m_end = m_start + pd.offsets.MonthEnd(1)
    monthly_rain = chirps.filterDate(m_start.strftime('%Y-%m-%d'), m_end.strftime('%Y-%m-%d')).sum()
    val = monthly_rain.reduceRegion(ee.Reducer.mean(), catchment_region, scale=5000, bestEffort=True).get('precipitation')
    rainfall_records.append({'month': m_start, 'rainfall_mm': val.getInfo() if val is not None else np.nan})

rain_df = pd.DataFrame(rainfall_records).set_index('month')

merged = ts_df[['NDCI_chla']].join(rain_df, how='inner').dropna()

# Lagged cross-correlation, 0-2 month lag (bloom response typically follows runoff pulses with a delay)
print("Lag (months) | Correlation (rainfall vs. NDCI)")
for lag in range(0, 3):
    if len(merged) > lag + 2:
        corr = merged['rainfall_mm'].shift(lag).corr(merged['NDCI_chla'])
        print(f"{lag:>12} | {corr:.3f}")

print("\n⚠️ Correlation, not causation — and n is limited by however many overlapping months you actually have.")
print("A weak or negative correlation is itself a real finding: it would argue against a simple runoff-driven")
print("mechanism and toward retention/internal loading as the dominant driver instead.")


### 12. Distance-Decay Regression — Discharge Points vs. Bloom Frequency

Stronger evidence than a flat buffer-mean per point: bins the whole gulf by distance from the nearest discharge
point and tests whether bloom frequency actually decays with distance. **Requires Section 9's discharge point data
and Section 8's bloom_freq_raster — both currently stubs.**


In [ ]:
def distance_decay_analysis(bloom_freq_raster, discharge_points_fc, aoi, distance_bins_m=None):
    """Bins bloom frequency by distance from nearest discharge point. Run once Sections 8-9 are complete."""
    if distance_bins_m is None:
        distance_bins_m = [0, 250, 500, 1000, 2000, 4000, 8000]

    distance_img = discharge_points_fc.distance(searchRadius=10000).clip(aoi)

    results = []
    for i in range(len(distance_bins_m) - 1):
        lo, hi = distance_bins_m[i], distance_bins_m[i + 1]
        band_mask = distance_img.gte(lo).And(distance_img.lt(hi))
        mean_freq = bloom_freq_raster.updateMask(band_mask).reduceRegion(
            ee.Reducer.mean(), aoi, scale=50, maxPixels=1e9, bestEffort=True
        ).get('bloom_frequency')
        results.append({'distance_band_m': f"{lo}-{hi}", 'mean_bloom_frequency': mean_freq.getInfo() if mean_freq else None})

    decay_df = pd.DataFrame(results)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(decay_df['distance_band_m'], decay_df['mean_bloom_frequency'], color='firebrick')
    ax.set_xlabel('Distance from nearest discharge point (m)')
    ax.set_ylabel('Mean bloom frequency')
    ax.set_title('Bloom Frequency vs. Distance from Registered Discharge Points')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('distance_decay.png', dpi=200)
    plt.show()
    return decay_df

# decay_df = distance_decay_analysis(bloom_freq_raster, discharge_points, aoi)
print("distance_decay_analysis() defined — run once discharge point data (Sec 9) and bloom_freq_raster (Sec 8) exist.")
print("⚠️ No decay with distance is a real, reportable finding — it points toward diffuse/informal pollution")
print("sources rather than registered point sources, not toward a failed analysis.")


## Tier 2 — Statistical Rigor & Mechanism

Turns visual impressions ("looks like it's increasing") into defensible statistical claims, and adds a
socioeconomic driver (land cover change) alongside the climatic one (rainfall) from Tier 1.


### 13. Mann-Kendall Trend Test

Non-parametric trend test appropriate for time series with gaps and non-normal distributions (both apply here) —
replaces "the chart appears to trend upward" with an actual trend statistic and significance level.


In [ ]:
def mann_kendall_test(series):
    """Returns (trend direction, p-value, Sen's slope) for a pandas Series indexed by date."""
    s = series.dropna()
    n = len(s)
    if n < 8:
        print(f"⚠️ Only {n} non-missing observations — Mann-Kendall needs a longer record to be reliable.")
        return None

    x = s.values
    S = sum(np.sign(x[j] - x[i]) for i in range(n) for j in range(i + 1, n))
    var_S = n * (n - 1) * (2 * n + 5) / 18
    z = (S - 1) / np.sqrt(var_S) if S > 0 else (S + 1) / np.sqrt(var_S) if S < 0 else 0

    from scipy.stats import norm
    p_value = 2 * (1 - norm.cdf(abs(z)))

    slopes = [(x[j] - x[i]) / (j - i) for i in range(n) for j in range(i + 1, n)]
    sen_slope = np.median(slopes)

    trend = "increasing" if S > 0 else "decreasing" if S < 0 else "no trend"
    return {'trend': trend, 'p_value': p_value, 'sen_slope_per_month': sen_slope, 'significant_at_0.05': p_value < 0.05}

mk_result = mann_kendall_test(ts_df['NDCI_chla'])
if mk_result:
    print(f"Trend: {mk_result['trend']}")
    print(f"Sen's slope: {mk_result['sen_slope_per_month']:.5f} NDCI units/month")
    print(f"p-value: {mk_result['p_value']:.4f} (significant at 0.05: {mk_result['significant_at_0.05']})")
    if not mk_result['significant_at_0.05']:
        print("⚠️ Not statistically significant — do not claim a confirmed long-term trend in your conclusions;")
        print("report it as a non-significant tendency, or state that the record is too short/gappy to resolve one.")


### 14. Catchment Land Cover Change (2018 → 2024)

Tracks cropland expansion in the Nyando/Kano Plains catchment using ESA WorldCover (2020, 2021) — this gives a
land-use driver story to pair with the rainfall mechanism test in Section 11, rather than leaving "agricultural
runoff" as an unverified assumption from the original project framing.

**⚠️ Data availability limitation:** ESA WorldCover currently has only two epochs (2020 and 2021) — it does **not**
cover your full 2018–2024 window. For a true 2018 vs. 2024 comparison you'd need Dynamic World (near-annual,
2018–present) instead; the cell below uses Dynamic World for that reason, with WorldCover as a higher-accuracy
cross-check for whichever years it does cover.


In [ ]:
def cropland_fraction(year, region):
    """Annual mode land cover classification from Dynamic World, cropland class fraction within region."""
    dw = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
          .filterBounds(region)
          .filterDate(f'{year}-01-01', f'{year}-12-31')
          .select('label'))
    if dw.size().getInfo() == 0:
        return None
    mode_classified = dw.mode()  # most frequent class per pixel across the year
    cropland_mask = mode_classified.eq(4)  # Dynamic World class 4 = crops
    stats = cropland_mask.reduceRegion(ee.Reducer.mean(), region, scale=30, maxPixels=1e9, bestEffort=True)
    return stats.get('label').getInfo()

catchment_2018 = cropland_fraction(2018, catchment_region)
catchment_2024 = cropland_fraction(2024, catchment_region)

print(f"Cropland fraction, catchment, 2018: {catchment_2018}")
print(f"Cropland fraction, catchment, 2024: {catchment_2024}")
if catchment_2018 and catchment_2024:
    change = (catchment_2024 - catchment_2018) / catchment_2018 * 100
    print(f"Relative change: {change:+.1f}%")
    print("⚠️ Pair this with Section 11's rainfall correlation, not in isolation — a real driver story needs both")
    print("the land-use change AND the runoff-response mechanism, not just a coincidental trend in each.")


### 15. Risk Classification Against Literature Thresholds

Re-expresses your K-means bloom severity classes against **published, Winam-Gulf-relevant** reference values,
so output is usable by a non-technical county official rather than only a GIS specialist.

**Citable sources found — two directly relevant, one general:**

1. **Ogembo, V., & Mohamed, G. A. (2023).** *Spectral Reflectance and Algal Bloom Monitoring of Lake Victoria
   Using Remote Sensing Techniques, Winam Gulf of Kenya.* Earth Sciences, 12(4), 90–98.
   `https://doi.org/10.11648/j.earth.20231204.12` — **this is the same water body, same NDCI method, same
   GEE-based approach you're using.** Reports mean Chl-a of **33–50 mg/m³ in Winam Gulf**, with peak
   concentrations in **January–February**, driven by dry-season heat rather than rainy-season runoff.
   ⚠️ **This revises the seasonality hypothesis in Section 11** — the original framing assumed a rains-driven
   runoff peak; this paper documents the opposite pattern for this specific gulf. Check your own rainfall
   cross-correlation results against this before writing your discussion section.

2. **Mishra, S., & Mishra, D. R. (2012)** — defines NDCI itself; anchors NDCI ≈ −0.1 (low Chl-a) and
   NDCI ≈ 0.5 (severe bloom), as applied by Digital Earth Africa (2021).

3. **UNEP (2023) / Carlson (1977) trophic state index**, as tabulated in Gimode et al. (2025, *African Journal
   of Aquatic Science*, Suguti Bay, Lake Victoria — same lake, Tanzanian side): Oligotrophic ≤ 8.0 mg/m³,
   Mesotrophic 8–25, Eutrophic 25–75, Hypereutrophic ≥ 75.0 mg/m³. Cross-checking against source (1): a
   Winam Gulf mean of 33–50 mg/m³ places the gulf **solidly in the Eutrophic category** on this scale.

4. **WHO (2022) cyanobacterial Alert Levels**, also tabulated in Gimode et al. (2025): AL1 = 1 µg/L Chl-a
   (lifetime provisional guideline value), AL2 = 12 µg/L Chl-a (short-term provisional guideline value).

**⚠️ What this still doesn't give you:** none of these sources publish an NDCI-to-mg/m³ regression specific to
Winam Gulf — source (1) reports a mean concentration alongside NDCI use, not a fitted conversion equation, and
the Gimode paper's OC3 calibration is for Suguti Bay, a different bay on different DE Africa infrastructure. The
`threshold_table` below therefore does two separate things, kept clearly apart: an NDCI-based severity scale
(sources 2) and a concentration-based trophic/health context (sources 1, 3, 4) that you can *report alongside*
your NDCI results as literature context — not claim your NDCI values directly equal.


In [ ]:
# --- NDCI severity scale (Mishra & Mishra, 2012; DE Africa, 2021) ---
# Low/severe anchors are cited; 'moderate'/'elevated' are linearly interpolated, not independently published.
threshold_table = {
    'low':      (-1.00, -0.10, 'Mishra & Mishra (2012); DE Africa (2021) — cited anchor'),
    'moderate': (-0.10,  0.15, 'interpolated between cited anchors — not independently published'),
    'elevated': ( 0.15,  0.50, 'interpolated between cited anchors — not independently published'),
    'severe':   ( 0.50,  1.00, 'Mishra & Mishra (2012); DE Africa (2021) — cited anchor: severe bloom event'),
}

def classify_risk(ndci_value, threshold_table=threshold_table):
    for category, (lo, hi, source) in threshold_table.items():
        if lo <= ndci_value < hi:
            return category, source
    return "unclassified", None

ts_df[['risk_category', 'risk_source']] = ts_df['NDCI_chla'].apply(
    lambda v: pd.Series(classify_risk(v)) if pd.notnull(v) else pd.Series([None, None])
)
print(ts_df[['NDCI_chla', 'risk_category']].tail(12))

# --- Winam-Gulf-specific concentration context (report alongside, don't equate to your NDCI values) ---
WINAM_GULF_REPORTED_MEAN_CHLA_MGM3 = (33, 50)   # Ogembo & Mohamed (2023), Winam Gulf, NDCI-based study
WINAM_GULF_PEAK_MONTHS = ['January', 'February']  # same source — dry-season peak, NOT rains-driven

TROPHIC_BANDS_MGM3 = {  # UNEP (2023)/Carlson (1977), via Gimode et al. (2025)
    'Oligotrophic': (0, 8.0),
    'Mesotrophic':  (8.0, 25.0),
    'Eutrophic':    (25.0, 75.0),
    'Hypereutrophic': (75.0, float('inf')),
}
WHO_ALERT_LEVELS_UGL = {'AL1': 1, 'AL2': 12}  # WHO (2022), via Gimode et al. (2025)

gulf_mean = sum(WINAM_GULF_REPORTED_MEAN_CHLA_MGM3) / 2
trophic_status = next(name for name, (lo, hi) in TROPHIC_BANDS_MGM3.items() if lo <= gulf_mean < hi)
print(f"\nReported Winam Gulf mean Chl-a ({gulf_mean:.0f} mg/m3) falls in the '{trophic_status}' trophic band")
print("(UNEP/Carlson scale). This is literature context for your discussion section, not a value derived")
print("from this notebook's own NDCI output — keep the two clearly distinguished when you write it up.")

# Sanity check against your own peak months, once ts_df is fully populated:
# your_peak_months = ts_df.groupby(ts_df.index.month)['NDCI_chla'].mean().idxmax()
# print(f"Your data's peak NDCI month: {your_peak_months} — compare against Jan/Feb reported above.")


## Tier 3 — Operational Framing

Moves the notebook from a one-time retrospective analysis toward something a county office could plausibly run
on a recurring basis.


### 16. Seasonal Bloom Forecast

A lightweight seasonal-naive forecast (not a claim of predictive skill beyond what a short, gappy record supports)
— gives an early-warning framing rather than only a retrospective one. Treat this as illustrative unless validated
against held-out months.


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

def simple_seasonal_forecast(series, periods_ahead=3):
    s = series.dropna()
    if len(s) < 24:
        print(f"⚠️ Only {len(s)} months available — seasonal decomposition needs at least ~2 full annual cycles")
        print("to separate seasonality from noise reliably. Treat any forecast from a shorter series as indicative only.")
        return None

    s_monthly = s.asfreq('MS')  # explicit monthly frequency; gaps will show as NaN, not silently dropped
    decomposition = seasonal_decompose(s_monthly.interpolate(), model='additive', period=12)

    seasonal_pattern = decomposition.seasonal[-12:]
    trend_last = decomposition.trend.dropna().iloc[-1]

    forecast_index = pd.date_range(s.index[-1] + pd.DateOffset(months=1), periods=periods_ahead, freq='MS')
    forecast = [trend_last + seasonal_pattern.iloc[i % 12] for i in range(periods_ahead)]

    forecast_df = pd.DataFrame({'forecast_NDCI': forecast}, index=forecast_index)

    fig, ax = plt.subplots(figsize=(12, 5))
    s.plot(ax=ax, label='Observed', color='green')
    forecast_df['forecast_NDCI'].plot(ax=ax, label='Forecast (seasonal-naive)', color='orange', linestyle='--', marker='o')
    ax.set_title('Winam Gulf Chl-a Proxy — Observed vs. Seasonal-Naive Forecast')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('bloom_forecast.png', dpi=200)
    plt.show()

    print("⚠️ This is a seasonal-naive baseline, not a validated predictive model. Report it as illustrative of")
    print("expected seasonal pattern, not as a forecast with quantified confidence, unless you back-test it")
    print("against held-out months first.")
    return forecast_df

forecast_df = simple_seasonal_forecast(ts_df['NDCI_chla'])


### 17. Auto-Generated One-Page Monitoring Report

Pulls the latest month's index values, risk classification (if Section 15 is populated), and time-series context
into a single-page PDF — the kind of artifact a county office could actually receive and act on monthly, rather
than a notebook they'd need a GIS specialist to interpret.


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

def generate_monthly_report(ts_df, output_path='winam_gulf_monthly_report.pdf'):
    if ts_df.empty:
        print("No data to report — run Section 6 first.")
        return

    latest = ts_df.iloc[-1]
    latest_month = ts_df.index[-1].strftime('%B %Y')

    with PdfPages(output_path) as pdf:
        fig = plt.figure(figsize=(8.5, 11))
        fig.suptitle(f'Winam Gulf Water Quality — Monitoring Brief: {latest_month}', fontsize=14, fontweight='bold')

        ax1 = fig.add_axes([0.1, 0.65, 0.8, 0.22])
        ts_df['NDCI_chla'].plot(ax=ax1, color='green')
        ax1.set_title('Chlorophyll-a Proxy (NDCI) — Full Record', fontsize=10)
        ax1.grid(alpha=0.3)

        ax2 = fig.add_axes([0.1, 0.35, 0.8, 0.22])
        ts_df['TSS_proxy'].plot(ax=ax2, color='saddlebrown')
        ax2.set_title('TSS Proxy — Full Record', fontsize=10)
        ax2.grid(alpha=0.3)

        summary_text = (
            f"Latest month: {latest_month}\n"
            f"NDCI (Chl-a proxy): {latest['NDCI_chla']:.3f}\n"
            f"TSS proxy: {latest['TSS_proxy']:.3f}\n"
            f"CDOM proxy: {latest['CDOM_proxy']:.3f}\n"
            f"Scenes used this month: {int(latest['n_scenes'])}\n\n"
            "Note: values are relative satellite-derived indices, not calibrated concentrations.\n"
            "See accompanying methods documentation for index definitions and limitations."
        )
        fig.text(0.1, 0.08, summary_text, fontsize=9, va='bottom', family='monospace')

        pdf.savefig(fig)
        plt.close(fig)

    print(f"Report saved: {output_path}")

generate_monthly_report(ts_df)


# Part III — Interpretation, Exports, and Honest Limitations

## Interpretation Guidance

1. **Baseline comparison (Sec 10):** Report the gulf-vs-reference ratio with its exact overlap period — don't
   extrapolate beyond the months you actually compared.
2. **Rainfall mechanism (Sec 11):** A strong positive lagged correlation supports runoff-driven eutrophication;
   a weak one supports retention/internal nutrient cycling as the dominant driver instead — both are real findings.
3. **Distance-decay (Sec 12):** No decay = evidence for diffuse/informal pollution sources, not a failed analysis.
4. **Trend significance (Sec 13):** Only claim "worsening" in your conclusions if Mann-Kendall came back significant
   at p < 0.05 — otherwise report it as a non-significant tendency.
5. **Land cover (Sec 14):** Only claims a driver relationship when paired with Section 11's rainfall-response test —
   parallel trends alone aren't a mechanism.
6. **Risk classification (Sec 15):** Only usable once you've inserted and cited real threshold values — do not
   ship the placeholder structure as if it were populated.


## Exports

In [ ]:
ts_df.to_csv('winam_gulf_water_quality_timeseries.csv')
comparison_df.to_csv('winam_gulf_vs_reference_timeseries.csv') if 'comparison_df' in dir() else None
print("CSV exports saved locally.")

if monthly_composites:
    export_image = monthly_composites[-1].select(['NDCI_chla', 'TSS_proxy', 'CDOM_proxy'])
    task = ee.batch.Export.image.toDrive(
        image=export_image, description='winam_gulf_wq_index_latest',
        folder='WinamGulf_WQ_Outputs', region=aoi, scale=20, maxPixels=1e9
    )
    task.start()
    print("Export task started — monitor at https://code.earthengine.google.com/tasks")


## Summary of Flags to Resolve Before This Goes Into Your Final Report

| # | Section | Issue | Action Needed |
|---|---|---|---|
| 1 | 2 | AOI is derived from real water data but cutline placement is approximate | Visually inspect the rendered map; swap in an authoritative shapefile if you obtain one |
| 2 | 4 | Sun-glint correction is simplified (fixed-slope, not regression-based) | Validate against ENVI-based method if glint looks significant |
| 3 | 5 | Proxies are relative indices, not calibrated concentrations | Don't report as absolute mg/m³ without in-situ regression |
| 4 | 7 | K-means cluster count (k=4) is a judgment call | Cross-check against Section 15's literature thresholds once populated |
| 5 | 9, 12 | NEMA discharge data not included | Source from NEMA Kisumu (see earlier email), required for distance-decay analysis |
| 6 | 11, 14 | Catchment boundary is approximate, not a true watershed delineation | Refine with HydroSHEDS basin polygons if precision matters |
| 7 | 13 | Trend claims require p < 0.05 | Report Mann-Kendall result honestly, including non-significant outcomes |
| 8 | 15 | Risk thresholds now cite Mishra & Mishra (2012)/DE Africa for the low/severe anchors; middle bins are interpolated | State the interpolation explicitly in your methods — don't cite midpoints as independently published |
| 9 | 16 | Forecast is seasonal-naive, not validated | Back-test against held-out months before claiming predictive skill |
| 10 | 5b | In-situ calibration uses Ugandan waters, not Winam Gulf directly | Report as regional proxy calibration; verify Zenodo column names before running |
| 11 | 5b | Calibration regression has a fit-quality gate (R²>0.5, n≥10) | Do not use the regression if it fails the gate — report attempted-but-inconclusive instead |
| 12 | 6 | Median composite alone understates peak bloom severity | Now reports median AND max NDCI per month — cite the correct one per claim |
| 13 | 6 | Months built from very few scenes were previously weighted equally | Now flagged as low-confidence (≤2 scenes) and shaded in the time-series plot |
| 14 | 5c | Landsat 8 has no red-edge band — NDCI/Chl-a cannot be computed from it | Only TSS/CDOM can be gap-filled with L8; Chl-a stays Sentinel-2-only |
| 15 | 5c | S2/L8 cross-calibration has the same fit-quality gate as Section 5b | Do not blend sensors into one series unless R²>0.5 and n≥10 |
| 10 | 5b | In-situ calibration uses Ugandan Lake Victoria stations, not Winam-Gulf-specific samples | Report as lake-wide approximation; get real Winam Gulf field samples if precision matters |
| 11 | 6 | Monthly compositing now reports mean/median/max/std, not mean alone | Use *_max columns for peak-bloom claims, not just *_mean, when writing results |

This notebook is a working analytical scaffold. The flags above are the actual judgment calls that determine
whether your conclusions hold up — resolve them deliberately.
